# Raymond RLHF-PPO 训练

**RLHF 流程中的第二步：**
```
SFT model  →  [reward_model_train.ipynb] Reward Model  →  [本脚本] PPO 训练
```

**PPO 四个关键角色：**

| 角色 | 作用 | 初始化自 | 是否训练 |
|---|---|---|---|
| **Actor（Policy）** | 生成回复的主模型 | SFT model | ✅ 是（LoRA） |
| **Critic（Value）** | 估计状态价值，辅助 PPO 更新 | Reward Model | ✅ 是（LoRA） |
| **Reward Model（RM）** | 给回复打分 | RM checkpoint | ❌ 冻结 |
| **Reference（Ref）** | KL 散度基准，防止跑偏 | SFT model | ❌ 冻结 |

**PPO 目标函数：**
$$\mathcal{J}_{\text{PPO}} = \mathbb{E}\left[r_\theta(x, y) - \beta \cdot \text{KL}(\pi_\theta(y|x) \| \pi_{\text{ref}}(y|x))\right]$$

- $r_\theta$：Reward Model 给出的分数
- KL 项：Actor 不能和 Ref 偏离太远（防止奖励 hacking）
- $\beta$：KL 惩罚系数，通常 0.01~0.1

**PPO vs DPO 选哪个？**
- 先试 DPO，效果不理想再上 PPO
- PPO 调参复杂，但理论上对「在线探索」更有优势

| 参数 | T4 | A100 | H100 |
|---|---|---|---|
| ppo_buffer_size | 4 | 8 | 16 |
| learning_rate | 1e-6 | 1e-6 | 5e-7 |
| 预计时长 | ~60分钟 | ~25分钟 | ~10分钟 |

## Cell 1：安装依赖

In [ ]:
!pip install -q llamafactory
!llamafactory-cli version

## Cell 2：挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

# SFT 原始数据（PPO 的 prompt 来源）
SFT_DATA_PATH = '/content/drive/MyDrive/raymond/raymond_train.json'
# SFT merged model（Actor 和 Ref 初始化）
SFT_MODEL_PATH = '/content/drive/MyDrive/raymond/merged_model'
# 奖励模型（需要先运行 reward_model_train.ipynb）
RM_PATH = '/content/drive/MyDrive/raymond/reward_model'

for path, name in [(SFT_DATA_PATH, 'SFT 数据'), (SFT_MODEL_PATH, 'SFT 模型'), (RM_PATH, '奖励模型')]:
    if os.path.exists(path):
        print(f'✓ {name}: {path}')
    else:
        print(f'✗ 找不到 {name}: {path}')

## Cell 3：准备 PPO 的 Prompt 数据集

PPO 训练不需要 chosen/rejected，只需要 **prompt**（对话历史）。
模型在线生成回复，RM 实时打分，PPO 据此更新 Actor。

我们从 SFT 训练数据中提取对话历史作为 prompt。

In [ ]:
import os, json

DATA_DIR = '/content/llama_factory_data'
os.makedirs(DATA_DIR, exist_ok=True)

with open(SFT_DATA_PATH) as f:
    sft_data = json.load(f)

# 从 SFT 数据中提取 prompt（去掉最后一轮 gpt 回复）
# PPO 格式：只需要到 human 为止的对话历史
ppo_prompts = []
for sample in sft_data:
    convs = sample.get('conversations', [])
    # 找到最后一个 gpt 消息前的所有消息
    prompt_convs = []
    for msg in convs:
        if msg['from'] == 'gpt':
            # 把最后一个 human 消息前的内容作为 prompt
            break
        prompt_convs.append(msg)
    # 重新按轮次切分，取合理长度的 prompt
    if len(convs) >= 2:
        # 取前几轮 + 最后一个 human 消息
        last_human_idx = max(i for i, m in enumerate(convs) if m['from'] == 'human')
        prompt_convs = convs[:last_human_idx + 1]
        if len(prompt_convs) >= 2:  # 至少要有 system + human
            ppo_prompts.append({'conversations': prompt_convs})

# 采样（PPO 不需要那么多 prompt，500 条足够）
import random
random.seed(42)
if len(ppo_prompts) > 500:
    ppo_prompts = random.sample(ppo_prompts, 500)

with open(f'{DATA_DIR}/raymond_ppo_prompts.json', 'w') as f:
    json.dump(ppo_prompts, f, ensure_ascii=False)

dataset_info = {
    'raymond_ppo': {
        'file_name': 'raymond_ppo_prompts.json',
        'formatting': 'sharegpt',
        'columns': {'messages': 'conversations'},
        'tags': {'role_tag': 'from', 'content_tag': 'value',
                 'user_tag': 'human', 'assistant_tag': 'gpt', 'system_tag': 'system'}
    }
}
with open(f'{DATA_DIR}/dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)

print(f'PPO prompt 数据：{len(ppo_prompts)} 条')
print('数据目录:', os.listdir(DATA_DIR))

## Cell 4：检查 GPU

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {name} | 显存: {mem:.1f} GB')
    if mem < 20:
        print('⚠️  警告：PPO 需要同时加载 Actor + Ref + RM，T4 显存较紧张')
        print('   建议使用 A100 或在 T4 上开启 4bit 量化')

## Cell 5：生成 PPO 训练配置

**PPO 关键超参说明：**
- `ppo_buffer_size`：每轮 PPO 更新收集的样本数（越大越稳定，但越慢）
- `ppo_epochs`：每批数据做几次 PPO 更新（通常 1-4）
- `kl_coef`（β）：KL 散度惩罚系数，防止偏离 Ref 太远
- `reward_model`：已训练的奖励模型路径

In [ ]:
import torch, yaml, os

PPO_OUTPUT_DIR = '/content/drive/MyDrive/raymond/ppo_output'
os.makedirs(PPO_OUTPUT_DIR, exist_ok=True)

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if mem_gb >= 70:  # H100
    quant_config = {}
    batch_size = 4
    ppo_buffer_size = 16
    learning_rate = 5e-7
elif mem_gb >= 35:  # A100
    quant_config = {}
    batch_size = 2
    ppo_buffer_size = 8
    learning_rate = 1e-6
else:  # T4
    quant_config = {'quantization_bit': 4, 'quantization_method': 'bnb'}
    batch_size = 1
    ppo_buffer_size = 4
    learning_rate = 1e-6

ppo_config = {
    # Actor（Policy）：从 SFT 模型初始化
    'model_name_or_path': SFT_MODEL_PATH,
    'template': 'qwen3_nothink',
    'trust_remote_code': True,
    'flash_attn': 'auto',

    # Reward Model：冻结，只用于打分
    'reward_model': RM_PATH,
    'reward_model_type': 'lora',  # RM 是 LoRA adapter

    # 数据（只需要 prompts）
    'dataset': 'raymond_ppo',
    'dataset_dir': '/content/llama_factory_data',
    'cutoff_len': 1024,
    'preprocessing_num_workers': 4,

    # PPO 核心配置
    'stage': 'ppo',              # PPO 强化学习阶段
    'do_train': True,
    'ppo_buffer_size': ppo_buffer_size,  # 每轮收集的样本数
    'ppo_epochs': 2,             # 每批数据的 PPO 更新次数
    'ppo_score_norm': True,      # 归一化 reward 分数（更稳定）
    'ppo_whiten_rewards': False, # 是否 whiten rewards
    'kl_coef': 0.05,             # KL 散度惩罚（Actor vs Ref），越小对齐越激进

    # Actor LoRA
    'finetuning_type': 'lora',
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_target': 'all',

    # 训练超参（PPO 比 SFT/DPO 学习率更小）
    'num_train_epochs': 2,
    'per_device_train_batch_size': batch_size,
    'gradient_accumulation_steps': 2,
    'learning_rate': learning_rate,
    'lr_scheduler_type': 'cosine',
    'warmup_steps': 10,
    'max_grad_norm': 1.0,
    'optim': 'adamw_torch',
    'bf16': True,

    # 生成参数（PPO 在线采样时用）
    'max_new_tokens': 150,
    'temperature': 0.8,
    'top_p': 0.9,

    'output_dir': PPO_OUTPUT_DIR,
    'logging_steps': 5,
    'save_steps': 50,
    'plot_loss': True,
    'report_to': 'none',
    **quant_config,
}

ppo_config_path = '/content/ppo_config.yaml'
with open(ppo_config_path, 'w') as f:
    yaml.dump(ppo_config, f, default_flow_style=False, allow_unicode=True)

print('PPO 训练配置已生成')
print(f'kl_coef={ppo_config["kl_coef"]}, ppo_buffer_size={ppo_buffer_size}')
print(f'learning_rate={learning_rate}')
print()
print('模型加载顺序（同时在显存中）：')
print('  1. Actor (trainable LoRA)  - SFT model')
print('  2. Ref   (frozen)          - SFT model')
print('  3. RM    (frozen)          - Reward model')
print('  注意：T4 24GB 显存可能较紧张，如果 OOM 请减小 batch_size')

## Cell 6：开始 PPO 训练

In [ ]:
!llamafactory-cli train /content/ppo_config.yaml

## Cell 7：评估 PPO 训练结果

In [ ]:
import os, json

print('PPO 输出文件:', os.listdir(PPO_OUTPUT_DIR))

# PPO 关键指标：reward 均值（越大越好），kl 散度（不能太大）
trainer_log = f'{PPO_OUTPUT_DIR}/trainer_log.jsonl'
if os.path.exists(trainer_log):
    logs = []
    with open(trainer_log) as f:
        for line in f:
            try:
                logs.append(json.loads(line))
            except:
                pass
    if logs:
        # 查看 reward 趋势
        rewards = [l.get('reward', None) for l in logs if l.get('reward') is not None]
        kls = [l.get('kl', None) for l in logs if l.get('kl') is not None]
        if rewards:
            print(f'初始 reward 均值: {rewards[0]:.4f}')
            print(f'最终 reward 均值: {rewards[-1]:.4f}')
            if rewards[-1] > rewards[0]:
                print('✓ reward 上升，PPO 对齐有效')
            else:
                print('✗ reward 没有上升，检查 RM 质量或 kl_coef')
        if kls:
            print(f'最终 KL 散度: {kls[-1]:.4f}')
            if kls[-1] > 10:
                print('⚠️  KL 过大，模型偏离 SFT 太多，增大 kl_coef')
            elif kls[-1] < 1:
                print('△ KL 很小，对齐可能不够，可以减小 kl_coef')

## Cell 8：对比 SFT vs PPO 效果

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_PATH, trust_remote_code=True)

print('加载 SFT 模型...')
sft_base = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_PATH, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)
sft_base.eval()

print('加载 PPO 模型...')
ppo_base = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_PATH, torch_dtype=torch.bfloat16,
    device_map='cpu', trust_remote_code=True  # 放 CPU 节省显存
)
ppo_model = PeftModel.from_pretrained(ppo_base, PPO_OUTPUT_DIR)
ppo_model = ppo_model.to('cuda')
ppo_model.eval()

SYSTEM = (
    '你是Raymond，在美国留学的中国研究生。你说话短而碎，每条1-15字，用\\n分隔多条消息。'
    '口头禅：66、哈、f、说白了、不好说。幽默方式是自嘲和反讽。'
)

def chat_with(model, user_input, max_new_tokens=150):
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': user_input}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.8, do_sample=True, top_p=0.9)
    prompt_len = inputs['input_ids'].shape[-1]
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

test_prompts = ['你后悔出国吗', '你觉得未来怎样', '今天心情咋样', '铲吗']
print('\n=== SFT vs PPO 对比 ===')
for q in test_prompts:
    print(f'\n问题: {q}')
    print(f'SFT:  {chat_with(sft_base, q)}')
    print(f'PPO:  {chat_with(ppo_model, q)}')

## Cell 9（训练完成后）：合并并保存 PPO 模型

In [ ]:
PPO_MERGED_DIR = '/content/drive/MyDrive/raymond/ppo_merged_model'
os.makedirs(PPO_MERGED_DIR, exist_ok=True)

print('合并 PPO LoRA adapter...')
merged = ppo_model.merge_and_unload()
merged.save_pretrained(PPO_MERGED_DIR)
tokenizer.save_pretrained(PPO_MERGED_DIR)
print(f'合并完成: {PPO_MERGED_DIR}')
print('下一步：用 llama.cpp 量化为 GGUF，替换 Ollama 中的模型文件')